In [82]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import AutoConfig  # for from_pretrained
from datasets import load_dataset
import pandas as pd
import os
import token_loss
import torch

from tqdm.notebook import tqdm

# see updated requirements.txt for others
HF_KEY = os.environ.get("HF_KEY")
assert len(HF_KEY), "Set HF_KEY"

In [6]:
# Load the data
print("Loading pile data")
subsets = token_loss.load_pile_samples(HF_KEY=HF_KEY)

Loading pile data


You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggin

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2387 > 2048). Running this sequence through the model will result in indexing errors


17 sequences over length 1024 (of 1300)


In [7]:
# Load ruan's model list
model_file = "ruan_models.csv"  #sys.argv[1]
assert os.path.exists(model_file)


# Configuration
model_df = pd.read_csv(model_file).sort_values('Model Size (B)')
model_list = model_df["Model"].tolist()  # smallest (non-cached) models first
out_path = os.path.expanduser("~/data/ruan_losses")
os.makedirs(out_path, exist_ok=True)

In [ ]:
# Check the model list for compatibility
compat_models = {}

for model_name in model_list:

   blocked = [
      # requires Triton, which is no longer in pip...
      'mosaicml/mpt-7b',
      'mosaicml/mpt-30b',
   ]
 
   if model_name in blocked:
      continue

   # Check the metadata... there are two places we might find the max embeddings from
   tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_KEY, trust_remote_code=True)
   config = AutoConfig.from_pretrained(model_name, token=HF_KEY, trust_remote_code=True)
   
   max_ctx = tokenizer.model_max_length

   if hasattr(config, "max_position_embeddings"):
      max_ctx = min(max_ctx, config.max_position_embeddings)
      
   tokenized_data = []
   for example in subsets:
      tokens = tokenizer.encode(example["text"])
      tokenized_data.append(tokens)
   tokenlen = max(map(len, tokenized_data))

   # Now we check whether this model type will be compatible
   compat_models[model_name] = tokenlen < max_ctx
   if tokenlen >= max_ctx:
      print(f"{model_name} is INCOMPATIBLE - it has a context of {max_ctx} and encded length was {tokenlen}")
   else:
      print(f"{model_name} is compatible.")

EleutherAI/pythia-70m-deduped is compatible.
cerebras/Cerebras-GPT-111M is compatible.


Token indices sequence length is longer than the specified maximum sequence length for this model (1044 > 1024). Running this sequence through the model will result in indexing errors


openai-community/gpt2 has a context of 1024 and encded length was 1044
openai-community/gpt2 is not compatible.
facebook/opt-125m is compatible.
EleutherAI/gpt-neo-125m is compatible.
HuggingFaceTB/SmolLM-135M is compatible.
EleutherAI/pythia-160m-deduped is compatible.
RWKV/rwkv-4-169m-pile is compatible.
cerebras/Cerebras-GPT-256M is compatible.
facebook/opt-350m is compatible.


Token indices sequence length is longer than the specified maximum sequence length for this model (1044 > 1024). Running this sequence through the model will result in indexing errors


openai-community/gpt2-medium has a context of 1024 and encded length was 1044
openai-community/gpt2-medium is not compatible.
HuggingFaceTB/SmolLM-360M is compatible.
EleutherAI/pythia-410m-deduped is compatible.
RWKV/rwkv-4-430m-pile is compatible.
Qwen/Qwen1.5-0.5B is compatible.
Qwen/Qwen2-0.5B is compatible.
h2oai/h2o-danube3-500m-base is compatible.
bigscience/bloom-560m is compatible.
facebook/xglm-564M is compatible.
cerebras/Cerebras-GPT-590M is compatible.


...

tiiuae/falcon-rw-1b has a context of 1024 and encded length was 1044
tiiuae/falcon-rw-1b is not compatible.
bigscience/bloom-1b1 is compatible.
TinyLlama/TinyLlama_v1.1 is compatible.
deepseek-ai/deepseek-coder-1.3b-base is compatible.
microsoft/phi-1_5 is compatible.
facebook/opt-1.3b is compatible.
EleutherAI/gpt-neo-1.3B is compatible.
cerebras/Cerebras-GPT-1.3B is compatible.
EleutherAI/pythia-1.4b-deduped is compatible.


Token indices sequence length is longer than the specified maximum sequence length for this model (1044 > 1024). Running this sequence through the model will result in indexing errors


openai-community/gpt2-xl has a context of 1024 and encded length was 1044
openai-community/gpt2-xl is not compatible.
RWKV/rwkv-4-1b5-pile is compatible.
Qwen/Qwen2-1.5B is compatible.
stabilityai/stablelm-2-1_6b is compatible.
facebook/xglm-1.7B is compatible.
HuggingFaceTB/SmolLM-1.7B is compatible.
Qwen/Qwen1.5-1.8B is compatible.
h2oai/h2o-danube-1.8b-base is compatible.
h2oai/h2o-danube2-1.8b-base is compatible.
google/gemma-2b is compatible.
google/gemma-2-2b is compatible.
cerebras/Cerebras-GPT-2.7B is compatible.
microsoft/phi-2 is compatible.
EleutherAI/gpt-neo-2.7B is compatible.
facebook/opt-2.7b is compatible.
EleutherAI/pythia-2.8b-deduped is compatible.
bigcode/starcoderbase-3b is compatible.
bigcode/starcoder2-3b is compatible.
cerebras/btlm-3b-8k-base is compatible.
openlm-research/open_llama_3b_v2 is compatible.
openlm-research/open_llama_3b is compatible.
stabilityai/stablelm-base-alpha-3b is compatible.
togethercomputer/RedPajama-INCITE-Base-3B-v1 is compatible.
RWKV

tiiuae/falcon-7b is compatible.
openlm-research/open_llama_7b is compatible.
RWKV/rwkv-4-7b-pile is compatible.
togethercomputer/RedPajama-INCITE-Base-7B-v0.1 is compatible.
internlm/internlm2-7b is compatible.
deepseek-ai/deepseek-llm-7b-base is compatible.
Deci/DeciLM-7B is compatible.
stabilityai/stablelm-base-alpha-7b-v2 is compatible.
bigcode/starcoderbase-7b is compatible.
LLM360/Amber is compatible.
stabilityai/stablelm-base-alpha-7b is compatible.
Qwen/Qwen-7B is compatible.
Qwen/Qwen1.5-7B is compatible.
meta-llama/Llama-2-7b-hf is compatible.
allenai/OLMo-7B-0424-hf is compatible.
google/gemma-7b is compatible.
Qwen/Qwen2-7B is compatible.
codellama/CodeLlama-7b-hf is compatible.
bigscience/bloom-7b1 is compatible.
mistralai/Mistral-7B-v0.1 is compatible.
facebook/xglm-7.5B is compatible.
meta-llama/Meta-Llama-3-8B is compatible.
meta-llama/Meta-Llama-3.1-8B is compatible.
google/gemma-2-9b is compatible.
01-ai/Yi-1.5-9B is compatible.
ai21labs/Jamba-v0.1 is compatible.
mistr

tiiuae/falcon-40b is compatible.
mistralai/Mixtral-8x7B-v0.1 is compatible.
huggyllama/llama-65b is compatible.
facebook/opt-66b is compatible.
deepseek-ai/deepseek-llm-67b-base is compatible.
meta-llama/Meta-Llama-3.1-70B is compatible.
meta-llama/Llama-2-70b-hf is compatible.
codellama/CodeLlama-70b-hf is compatible.
meta-llama/Meta-Llama-3-70B is compatible.
Qwen/Qwen1.5-72B is compatible.
Qwen/Qwen-72B is compatible.
Qwen/Qwen2-72B is compatible.
Qwen/Qwen1.5-110B is compatible.
bigscience/bloom is compatible.
tiiuae/falcon-180B is compatible.
meta-llama/Meta-Llama-3.1-405B-FP8 is compatible.


In [41]:
# len([k for k,v in compat_models.items() if v])
# [k for k,v in compat_models.items() if not v]
valid = [k for k,v in compat_models.items() if v]  # num successful models!
print(f"There are {len(valid)} compatible models.")

There are 141 compatible models.


In [39]:
import json

with open('model_list.json', "w") as f:
    json.dump(valid, f, indent=2)
print("Saved model list")

Saved model list


In [ ]:
hf_name = "facebook/opt-125m"
device = "mps"

tokenizer = AutoTokenizer.from_pretrained(hf_name, token=HF_KEY, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    hf_name, revision=None,
    #device_map="auto",  # {"": "mps"},  # "auto",
    torch_dtype="auto",
    token=HF_KEY,
    trust_remote_code=True,
)
model = model.to(device)
model.eval()  # put in evaluate mode
print("model loaded")

model loaded


In [33]:
import numpy as np

In [83]:
# Process dataset in batches
batch_size = 32
all_losses = []
dataset = subsets

output = []

for i in tqdm(range(0, len(dataset), batch_size)):
    batch = dataset[i:i + batch_size]

    # Tokenize
    inputs = tokenizer(
        batch["text"],
        padding=True,
        add_special_tokens=False,
        return_tensors="pt"  # ensors
    )

    # Compute losses
    input_dev = inputs["input_ids"].to(device)
    losses = token_loss.compute_token_loss(model, input_dev)

    # Package output
    inputs_np = inputs["input_ids"].numpy()
    cuts = inputs["attention_mask"].numpy().sum(axis=1)
    for input, loss, cut, subset, idx in zip(inputs_np, losses, cuts, batch['subset'], batch['idx']):
        output.append(dict(
            tokens=tokenizer.batch_decode(input)[:cut],
            losses=loss[:cut],
            subset=subset,
            idx = idx,
        ))


  0%|          | 0/41 [00:00<?, ?it/s]

In [87]:
import pickle

with open("test.pkl", "wb") as f:
    pickle.dump(output, f)